In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
from sklearn.linear_model import LinearRegression
import glob
import scipy
from functools import reduce

from models_generation_utils import *
from comparaison_crAss_PMMoV_utils import *

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.ticker as ticker


In [ ]:
wwtp = 'CLICHY'

pmmov_crass_files = glob.glob('../../outputs/files/viral_data/PMMoV (Pepper mild mottle virus).csv')
pmmov_crass_files += glob.glob('../../outputs/files/viral_data/Crassphage.csv')
filepath = '../../data/Observatory_paper_mendeley.xlsx'

In [ ]:
pmmov_crass_coefs = {}

pmmov_crass_files_dict_ww = {}
pmmov_crass_files_dict_pop = {}

s1_dict = {}
s2_dict = {}

for file in pmmov_crass_files:
    print(file)
    
    this_key = file.split('.csv')[0].split('/')[-1]

    data_cp, pop_file, s1, s2 = main(filepath, file, wwtp)
    corr_our_model_phages, corr_vn_phages, corr_our_model_vn, corr_our_model_vn_smoothed = perform_correlation_computation(s1, s2)

    pmmov_crass_files_dict_ww[this_key] = data_cp
    pmmov_crass_files_dict_pop[this_key] = pop_file
    
    coef = s1.muX.mean() / s2.Nt_hat.mean()
    pmmov_crass_coefs[this_key] = coef
    s1_dict[this_key] = s1
    s2_dict[this_key] = s2

In [ ]:
this_key = 'Crassphage'
s1_temp = s1_dict[this_key]
s2_temp = s2_dict[this_key]

s1_temp = s1_temp.loc[(s1_temp.dateStart<='2023-01-01') | (s1_temp.dateStart>='2023-05-01')]
s2_temp = s2_temp.loc[(s2_temp.dateStart<='2023-01-01') | (s2_temp.dateStart>='2023-05-01')]

corr_our_model_phages, corr_vn_phages, corr_our_model_vn, corr_our_model_vn_smoothed = perform_correlation_computation(s1_temp, s2_temp)
print(f'Correlation between our population estimation and crAssphages: {corr_our_model_phages}.')
print(f'Correlation between van Nuijs et al.\'s original estimation and crAssphages: {corr_vn_phages}.')
print(f'Correlation between van Nuijs et al.\'s original estimation and our estimation: {corr_our_model_vn}.')
print(f'Correlation between van Nuijs et al.\'s modified estimation and our estimation: {corr_our_model_vn_smoothed}.')

In [ ]:
this_key = 'PMMoV (Pepper mild mottle virus)' 
pop_ref_sig = s2_dict[this_key]
pop_ref_sig = pop_ref_sig.loc[(pop_ref_sig.dateStart<='2023-01-01') | (pop_ref_sig.dateStart>='2023-05-01')]

In [ ]:
path_mes = '../../outputs/files/flow_data_MES_specific_calendar/MES.csv'

mes_SEC = pd.read_csv(path_mes, sep=";")
mes_SEC.dateStart = pd.to_datetime(mes_SEC.dateStart)
mes_SEC.muX = 10**(mes_SEC.muX)
mes_SEC = mes_SEC.loc[mes_SEC.dateStart.isin(pop_ref_sig.dateStart.tolist())]

In [ ]:
np.corrcoef(mes_SEC.muX.values, pop_ref_sig.Nt_hat.values)

In [ ]:
temp_s1 = s1_dict['PMMoV (Pepper mild mottle virus)']
temp_s2 = s2_dict['PMMoV (Pepper mild mottle virus)']
temp_s1.muX.mean() / temp_s2.Nt_hat.mean()

In [ ]:
temp_s1 = s1_dict['PMMoV (Pepper mild mottle virus)'].muX.values
temp_s2 = s2_dict['PMMoV (Pepper mild mottle virus)'].Nt_hat.values

ratio = temp_s1 / temp_s2

In [ ]:
np.percentile(ratio, 2.5)*1e-10

In [ ]:
np.percentile(ratio, 97.5)*1e-11

In [ ]:
np.mean(ratio)*1e-10

In [ ]:
temp_s1 = s1_dict['Crassphage'].muX.values
temp_s2 = s2_dict['Crassphage'].Nt_hat.values

ratio = temp_s1 / temp_s2

In [ ]:
np.percentile(ratio, 2.5)*1e-10

In [ ]:
np.percentile(ratio, 97.5)*1e-11

In [ ]:
np.mean(ratio)*1e-10

In [ ]:
pmmov_crass_coefs

In [ ]:
with plt.style.context(['science', 'notebook', 'grid']):

    ratio_factor = 1#0.875
    KEY_SIZE = int(40 * ratio_factor)
    LABEL_SIZE = int(40 * ratio_factor)
    TICK_SIZE = int(40 * ratio_factor)
    TITLE_SIZE = int(46 * ratio_factor)
    LEGEND_SIZE = int(36 * ratio_factor)
    DATES_SIZE = 18
    figsize = (32, 8) #figsize = (32, 10)
    #figsize = (28, 6) #figsize = (32, 10)
    
    plt.rc('axes', labelsize=LABEL_SIZE)
    plt.rc('xtick', labelsize=TICK_SIZE)   
    plt.rc('ytick', labelsize=TICK_SIZE)
    plt.rc('figure', titlesize=TITLE_SIZE)
    plt.rc('legend', fontsize=LEGEND_SIZE)
    plt.rcParams['text.usetex'] = True
    
    fig = plt.figure(figsize=figsize, layout="constrained")
    
    ax_dict = fig.subplot_mosaic(
        """
        AB
        """
    )
    
    ######################################################### A #########################################################
    sub_data_iage = pmmov_crass_files_dict_pop['PMMoV (Pepper mild mottle virus)'].copy()
    this_molecule = 'Clichy'
    color = 'orange'
    this_letter = 'A'
    ax_dict[this_letter].plot(sub_data_iage.dateStart.values, sub_data_iage.Nt_hat.values, label='Estimated population', color=color, linewidth=10, zorder=3)
    ax_dict[this_letter].plot(sub_data_iage.dateStart.values, sub_data_iage.Nt_hat.values, color='black', linewidth=3, zorder=3)
        
    
    # Cases
    cases_data_iage = pmmov_crass_files_dict_ww['PMMoV (Pepper mild mottle virus)'].copy()
    ax_cases_1 = ax_dict[this_letter].twinx()

    ax_cases_1.plot(cases_data_iage.dateStart.values, cases_data_iage.muX.values, color='forestgreen', label='PMMoV flow', linewidth=10, zorder=3)
    ax_cases_1.plot(cases_data_iage.dateStart.values, cases_data_iage.muX.values, color='black', linewidth=3, zorder=3)

    
    ax_dict[this_letter].set_ylabel("$N_t$")
    ax_dict[this_letter].set_xlabel("Date")
    ax_dict[this_letter].tick_params(axis='x', labelsize=TICK_SIZE, rotation=45)
    ax_dict[this_letter].tick_params(axis='y', labelsize=TICK_SIZE)
    ax_dict[this_letter].grid(linewidth=1, color='black', alpha=0.8)
    ax_dict[this_letter].set_title(this_molecule + ' - PMMoV', size=TITLE_SIZE)
    
    ax_cases_1.set_ylabel("Daily flow (GU.day$^{-1}$)")

    ######################################################### B #########################################################
    this_letter = 'B'
    sub_data_iage = pmmov_crass_files_dict_pop['Crassphage'].copy()
    this_molecule = 'Clichy'
    color = 'orange'
    ax_dict[this_letter].plot(sub_data_iage.dateStart.values, sub_data_iage.Nt_hat.values, label='Estimated population', color=color, linewidth=10, zorder=3)
    ax_dict[this_letter].plot(sub_data_iage.dateStart.values, sub_data_iage.Nt_hat.values, color='black', linewidth=3, zorder=3)
        
    
    # Cases
    cases_data_iage = pmmov_crass_files_dict_ww['Crassphage'].copy()
    ax_cases_B_1 = ax_dict[this_letter].twinx()

    ax_cases_B_1.plot(cases_data_iage.dateStart.values, cases_data_iage.muX.values, color='lightseagreen', label='CrAssphages flow', linewidth=10, zorder=3)
    ax_cases_B_1.plot(cases_data_iage.dateStart.values, cases_data_iage.muX.values, color='black', linewidth=3, zorder=3)

    
    ax_dict[this_letter].set_ylabel("$N_t$")
    ax_dict[this_letter].set_xlabel("Date")
    ax_dict[this_letter].tick_params(axis='x', labelsize=TICK_SIZE, rotation=45)
    ax_dict[this_letter].tick_params(axis='y', labelsize=TICK_SIZE)
    ax_dict[this_letter].grid(linewidth=1, color='black', alpha=0.8)
    ax_dict[this_letter].set_title(this_molecule + ' - crAssphages', size=TITLE_SIZE)
    
    ax_cases_B_1.set_ylabel("Daily flow (GU.day$^{-1}$)")


    

    # Display subplot keys
    plt.rcParams['text.usetex'] = False
    fig.canvas.draw()
    
    # Function to align text with the ylabel of a specific axis
    def align_text_with_ylabel(ax, text, fig, offset=0.175):
        ylabel = ax.yaxis.label
        bbox = ylabel.get_window_extent()
        bbox_fig = fig.transFigure.inverted().transform(bbox)
        ylabel_center_fig_x = (bbox_fig[0, 0] + bbox_fig[1, 0]) / 2
        ylabel_center_fig_y = (bbox_fig[0, 1] + bbox_fig[1, 1]) / 2
        fig.text(ylabel_center_fig_x, ylabel_center_fig_y + offset, text, ha='center', va='center', size=KEY_SIZE, weight='bold')

    # Align text with the ylabels for each subplot
    for n, (key, ax) in enumerate(ax_dict.items()):
        align_text_with_ylabel(ax, key, fig, offset=0.45)
    
    
    # Main legend
    plt.rcParams['text.usetex'] = False
    h1, l1 = ax_dict['A'].get_legend_handles_labels()
    h2, l2 = ax_cases_1.get_legend_handles_labels()
    h3, l3 = ax_cases_B_1.get_legend_handles_labels()
    fig.legend(h1+h2+h3, l1+l2+l3, loc='upper center', bbox_to_anchor=(0.5, 0), fancybox=True, shadow=True, ncol=5)
    plt.savefig("../../outputs/figs/2025-12-18_" + "crAss_PmmoV_" + "pop_norm.pdf", bbox_inches = 'tight')
    plt.savefig("../../outputs/figs/2025-12-18_" + "crAss_PmmoV_" + "pop_norm.jpg", bbox_inches = 'tight')
    
    plt.show()